# RAG Pipeline Test (Stages 3–6)

This notebook covers:
1. **Indexing** — PDF → chunks → embeddings → ChromaDB
2. **Search** — similarity search with the retriever (persisted index)

**Prerequisites:** `uv sync` and a PDF in `data/raw/` (e.g. `sample.pdf`).

The first indexing run may take a while (downloads `BAAI/bge-small-en-v1.5`).

## Configuration

In [2]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDF_PATH = ROOT / "data/raw/sample.pdf"
VECTOR_DB_PATH = ROOT / "data/vector_db"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 3

print(f"Project root: {ROOT}")
print(f"PDF: {PDF_PATH}")
print(f"Vector DB: {VECTOR_DB_PATH}")

Project root: d:\Github\research-workflow-agent
PDF: d:\Github\research-workflow-agent\data\raw\sample.pdf
Vector DB: d:\Github\research-workflow-agent\data\vector_db


---
## Part A — Index PDF

Run this section on the **first run** or when you want to reindex a document (`reset=True`).

In [3]:
from rag.loaders import load_pdf
from rag.splitter import split_documents
from rag.embeddings import ChunkEmbedder
from rag.vectorstore import ChromaVectorStore

print("1. Loading PDF...")
documents = load_pdf(PDF_PATH)
print(f"   Pages loaded: {len(documents)}")

d:\Github\research-workflow-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Loading PDF...
   Pages loaded: 15


In [4]:
print("2. Chunking...")
chunks = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"   Chunks produced: {len(chunks)}")
print(f"   Sample (200 chars): {chunks[0].page_content[:200]}...")

2. Chunking...
   Chunks produced: 52
   Sample (200 chars): Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...


In [5]:
print("3. Embeddings (first run may take a while — model download)...")
embedder = ChunkEmbedder()
vectors = embedder.embed_chunks(chunks)
print(f"   Vectors: {len(vectors)}")
print(f"   Dimension: {embedder.embedding_dimension}")
print(f"   First 5 dims of chunk 0: {vectors[0][:5]}")

3. Embeddings (first run may take a while — model download)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2860.84it/s]


   Vectors: 52
   Dimension: 384
   First 5 dims of chunk 0: [-0.047576677054166794, 0.0075242649763822556, -0.04240209236741066, -0.03461027145385742, -0.017087772488594055]


In [6]:
print("4. Indexing in ChromaDB...")
store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.create_collection(reset=True)
indexed = store.add_documents(chunks, vectors)
store.persist()
print(f"   Chunks indexed: {indexed}")
print(f"   Collection: {store.collection_name}")
print(f"   Persisted at: {VECTOR_DB_PATH.resolve()}")

4. Indexing in ChromaDB...
   Chunks indexed: 52
   Collection: book_research
   Persisted at: D:\Github\research-workflow-agent\data\vector_db


---
## Part B — Search existing index

Use this section when the index **already exists** (Part A). No reindexing required.

In [7]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()
print(f"Collection loaded: {store.collection_name} ({store.collection.count()} chunks)")

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

Collection loaded: book_research (52 chunks)


In [8]:
question = "What is the attention mechanism?"
results = retriever.invoke(question)

print(f"Question: {question}")
print(f"Results: {len(results)}")
print()

for i, doc in enumerate(results, start=1):
    distance = doc.metadata.get("distance")
    page = doc.metadata.get("page", "n/a")
    print(f"--- Result {i} | page={page} | distance={distance} ---")
    print(doc.page_content[:400])
    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2557.10it/s]


Question: What is the attention mechanism?
Results: 3

--- Result 1 | page=1 | distance=0.2771965265274048 ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
descr

--- Result 2 | page=2 | distance=0.30977463722229004 ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Result 3 | page=12 | distance=0.3191729187965393 ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registr

In [9]:
queries = [
    "What is self-attention?",
    "How does the transformer architecture work?",
    "What are the main results of the paper?",
]

for query in queries:
    print("=" * 60)
    print(f"Question: {query}")
    for doc in retriever.invoke(query):
        preview = doc.page_content[:150].replace("\n", " ")
        print(f"  [page {doc.metadata.get('page')}] {preview}...")
    print()

Question: What is self-attention?
  [page 1] in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difficult to learn dependencies between di...
  [page 4] The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder...
  [page 2] 3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and ...

Question: How does the transformer architecture work?
  [page 2] Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully c...
  [page 4] The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder...
  [page 1] In this work we propose the Transformer, a model architecture eschewing rec

In [10]:
# Change the question and top_k here
custom_query = "Explain multi-head attention"
custom_top_k = 5

for doc in retriever.invoke(custom_query, top_k=custom_top_k):
    print(doc.metadata)
    print(doc.page_content[:300])
    print("-" * 40)

{'author': '', 'source': 'D:\\Github\\research-workflow-agent\\data\\raw\\sample.pdf', 'keywords': '', 'title': '', 'page_label': '5', 'subject': '', 'total_pages': 15, 'trapped': '/False', 'creator': 'LaTeX with hyperref', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'producer': 'pdfTeX-1.40.25', 'creationdate': '2024-04-10T21:11:43+00:00', 'page': 4, 'moddate': '2024-04-10T21:11:43+00:00', 'distance': 0.2093416452407837}
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhib
----------------------------------------
{'moddate': '2024-04-10T21:11:43+00:00', 'title': '', 'subject': '', 'page': 4, 'creationdate': '2024-04-10T21:11:43+00:00', 'total_pages': 15, 'creator': 'LaTeX with 